# Persistent Coupled Neural PDE–ODE for Cross-Recording Music Forecasting — V2

## Abstract

We evaluate a persistent-state coupled neural PDE–ODE on unseen recordings with an eight-step future forecast. The global state evolves jointly with the spectral field throughout the horizon. We compare it with conventional neural forecasting architectures and three pretrained Hugging Face time-series models: IBM TTM-R2, Chronos-Bolt-Tiny and Chronos-T5-Tiny. Recording-level splits, onset proxy metrics, high-frequency error, uncertainty across seeds, and compute costs are reported.

## Proposed method

We act as an **experimental research agent**: we define the prediction/repair task, construct leakage-safe observations, execute the proposed continuous-time model, challenge it with matched baselines, and independently verify the output. The proposed method is always marked **PROPOSED** in the architecture and evaluation diagrams.

**Problem:** persistent-state eight-step cross-recording forecasting. The spectrogram is a time-dependent state $u(t)$; the compact global state $z(t)$ has its own ODE. In the coupled architecture, both states influence one another. In the WSP extension, an external open-weight model proposes selected updates to a frozen network, never directly deciding acceptance.

## Expected outcomes (hypotheses, not measurements)

A valid advantage would require lower held-out spectral and onset-related errors without unacceptable compute or high-frequency detail loss. We do not assume that a time-series model pretrained on hourly signals will generalize to 16-ms audio features.

**Null hypothesis:** the proposed method does not improve the relevant held-out metrics compared with strong baselines at a comparable compute budget. We will preserve negative outcomes, failures, and missing pretrained-model evaluations.

## Methodological steps

We execute the workflow in this order: prepare data → instantiate the proposed model → train/freeze or diagnose → generate alternatives → validate on disjoint observations → evaluate test records. An evaluation diagram must distinguish the method under investigation from existing baselines and show where test leakage is prevented.

### Diagrams and protocol

The method diagram highlights the **persistent coupled PDE–ODE**, the evaluation diagram places three pretrained forecasting models alongside the proposed method, and the verification diagram shows why test recordings cannot be used for model selection. The long-context condition is necessary to evaluate IBM TTM-R2 without fake padding.

### How to interpret the three method diagrams

**Method architecture:** the highlighted node identifies the *proposed scientific mechanism*, not a generic preprocessing step. The incoming arrows specify evidence or state needed for the method; outgoing arrows specify the predictions or edits being evaluated.

**Comparative evaluation:** proposed-method results are distinguished from baselines using a thick outline or hatching. A visually highlighted method is **not** an assumed winner. Its score must be measured on the same held-out target as the baselines.

**Verification and failure handling:** training diagnostics may inform candidate generation; validation decides acceptance; the test partition is reserved for final reporting. Failed model downloads, invalid outputs, and rejected repairs remain visible and are not replaced by fabricated scores.

These diagrams are executable Graphviz figures generated inside Colab; the paper-style explanations and executable experiments remain the primary deliverable.


In [ ]:
%pip -q install graphviz torchdiffeq soundfile scipy pandas matplotlib

In [ ]:
import os, math, time, random, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from torchdiffeq import odeint
from scipy.signal import stft, resample_poly
from scipy.io import wavfile
SEED=211100343
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:',device,'torch:',torch.__version__)


In [ ]:
# Three executable research diagrams, saved inside the Colab runtime.
# The thicker proposed-method border is intentionally distinct from baseline methods.
from IPython.display import display, SVG
from graphviz import Digraph

def research_diagram(name, nodes, edges, proposed, caption):
    dot=Digraph(name,format='svg')
    dot.attr(rankdir='LR',fontname='Helvetica',labelloc='t',label=caption,nodesep='0.35',ranksep='0.45')
    for key,label in nodes:
        if key in proposed:
            dot.node(key,label,shape='box',style='rounded,bold',color='#B45309',penwidth='3.5',
                     fillcolor='#FFF7ED',fontcolor='#7C2D12')
        else:
            dot.node(key,label,shape='box',style='rounded',color='#64748B')
    for u,v in edges: dot.edge(u,v)
    try:
        svg=dot.pipe(format='svg').decode('utf-8')
        display(SVG(svg));Path(name+'.svg').write_text(svg)
    except Exception as exc:
        print('Graphviz rendering unavailable:',exc,'\n',dot.source)

research_diagram('v2_method',
    [('audio','Audio / spectrogram'),('diagnose','Diagnostics / local errors'),
     ('local','Local PDE-inspired operator'),('global','Persistent or short-horizon ODE'),
     ('proposer','PROPOSED: model / weight proposer'),('solve','Differentiable integration'),
     ('verify','Independent verifier')],
    [('audio','diagnose'),('diagnose','local'),('diagnose','global'),
     ('local','proposer'),('global','proposer'),('proposer','solve'),('solve','verify')],
    ['local','global','proposer','solve'], 'METHOD: '+'Persistent coupled dynamics on unseen tracks')
research_diagram('v2_evaluation',
    [('data','Recordings / synthetic signals'),('split','Train → validation → test'),
     ('proposed','PROPOSED: Persistent PDE–ODE'),('base','Classical / neural baselines'),
     ('hf','3 Hugging Face models (where applicable)'),('gate','Validation / selection gate'),
     ('out','Held-out MSE, onsets, latency')],
    [('data','split'),('split','proposed'),('split','base'),('split','hf'),
     ('proposed','gate'),('base','gate'),('hf','gate'),('gate','out')],
    ['proposed'], 'EVALUATION: isolate the proposed method from benchmarks')
research_diagram('v2_decisions',
    [('frozen','Frozen candidate / fixed design'),('probe','Residual / sensitivity probe'),
     ('action','PROPOSED: targeted intervention'),('validation','Disjoint validation'),
     ('accepted','Accept only if improvement'),('rejected','Reject / record failure'),
     ('test','One-shot held-out evaluation')],
    [('frozen','probe'),('probe','action'),('action','validation'),
     ('validation','accepted'),('validation','rejected'),('accepted','test'),('rejected','test')],
    ['action'], 'VERIFICATION: no candidate accesses test labels during selection')

## Step 1 — Register the long-context, recording-disjoint protocol

The TTM-R2 512-96 checkpoint requires a genuine 512-step input. We therefore use **512 consecutive STFT frames of context and forecast eight frames** for all architectures. The 16-ms audio frame hop is drastically different from TTM's recommended minute/hour time scale; results measure out-of-distribution transfer, not in-domain benchmark performance.

For real audio, upload a ZIP of WAV files long enough for a 512-frame context; otherwise 12-second synthetic tracks are generated. The three-seed, three-epoch setting is a **quick-run demonstration**, not an adequate statistical analysis.

In [ ]:
# Configuration and optional real-recording ZIP upload.
import zipfile, tempfile, os, glob, random, gc
from scipy.signal import stft, resample_poly
from scipy.io import wavfile
from torch.utils.data import TensorDataset, DataLoader
BENCH_SEEDS=[11,23,37]   # change to >=10 seeds for a paper
BENCH_H=8
BENCH_CONTEXT=512
BENCH_EPOCHS=3
BENCH_BATCH=8
BENCH_BANDS=48
BENCH_SR=8000
BENCH_HOP=128
BENCH_NFFT=512
BENCH_MAX_RECORDINGS=24
BENCH_UPLOAD_ZIP=False
BENCH_ROOT=Path('/content/music_benchmark')
BENCH_ROOT.mkdir(parents=True,exist_ok=True)
if BENCH_UPLOAD_ZIP:
    from google.colab import files
    z=files.upload()
    zipname=next(iter(z))
    with zipfile.ZipFile(zipname) as archive:
        for item in archive.infolist():
            destination=(BENCH_ROOT/item.filename).resolve()
            if not destination.is_relative_to(BENCH_ROOT.resolve()):
                raise ValueError('Unsafe ZIP member path')
            if item.is_dir(): continue
            if not item.filename.lower().endswith('.wav'):continue
            destination.parent.mkdir(parents=True,exist_ok=True)
            destination.write_bytes(archive.read(item))
print('Benchmark configuration:',BENCH_H,'steps,',len(BENCH_SEEDS),'seeds')

## Step 2 — Generate features and split by recording

We compute standardized log-frequency-band magnitudes. Training-only statistics normalize all splits; each track occurs in exactly one split. Short recordings are explicitly rejected rather than padded into the 512-step checkpoint.

In [ ]:
def bench_synth(seed, instrument, tempo, seconds=12.0):
    rng=np.random.default_rng(seed)
    tt=np.arange(int(seconds*BENCH_SR))/BENCH_SR
    x=np.zeros_like(tt)
    base=rng.choice([110.,146.83,164.81,196.,220.,246.94,261.63])
    harmonic_profiles={
      'soft':[1,.15,.05,.02,.01],
      'bright':[1,.7,.45,.25,.13],
      'reed':[1,.8,.12,.5,.07],
      'plucked':[1,.45,.22,.1,.05],
    }
    harms=harmonic_profiles[instrument]
    beat=60/tempo
    for k,start in enumerate(np.arange(0,seconds,beat)):
        note=base*(2**(int(rng.choice([-5,-3,0,2,4,7]))/12))
        length=min(int(beat*BENCH_SR),len(tt)-int(start*BENCH_SR))
        if length<=0:continue
        local=np.arange(length)/BENCH_SR
        decay=3.2 if instrument=='plucked' else .7
        env=np.minimum(local*30,1)*np.exp(-decay*local)
        s=sum(amp*np.sin(2*np.pi*(h+1)*note*local)
              for h,amp in enumerate(harms))
        x[int(start*BENCH_SR):int(start*BENCH_SR)+length]+=env*s
    x+=.003*rng.standard_normal(len(x))
    return (x/(np.max(np.abs(x))+1e-8)).astype(np.float32)

def bench_wav(path):
    sr,a=wavfile.read(path)
    a=a.astype(np.float32)
    if a.ndim>1:a=a.mean(axis=1)
    a/=max(float(np.max(np.abs(a))),1e-8)
    if sr!=BENCH_SR:
        from math import gcd
        g=gcd(int(sr),BENCH_SR)
        a=resample_poly(a,BENCH_SR//g,int(sr)//g)
    return a[:12*BENCH_SR]

def bench_features(a):
    _,_,z=stft(a,fs=BENCH_SR,nperseg=BENCH_NFFT,
               noverlap=BENCH_NFFT-BENCH_HOP,boundary=None)
    mag=np.abs(z).astype(np.float32)
    # Equal-frequency aggregation of the first 240 non-DC bins.
    mag=mag[1:241].reshape(BENCH_BANDS,5,-1).mean(axis=1)
    return np.log1p(20*mag).T.astype(np.float32)

paths=sorted(BENCH_ROOT.rglob('*.wav'))[:BENCH_MAX_RECORDINGS]
recordings=[]
if paths:
    for path in paths:
        try:
            audio=bench_wav(path)
            if len(audio)<9*BENCH_SR:continue
            recordings.append({'id':str(path.relative_to(BENCH_ROOT)),
                               'label':path.parent.name,
                               'features':bench_features(audio)})
        except Exception as e:
            print('Skipping',path.name,str(e)[:100])
else:
    for i in range(24):
        instrument=['soft','bright','reed','plucked'][i%4]
        tempo=[72,92,118,145,168][(i//4)%5]
        recordings.append({'id':f'synth_{i:02d}','label':instrument,
                           'tempo':tempo,
                           'features':bench_features(bench_synth(i+200,instrument,tempo))})
print('Recording count:',len(recordings),'labels:',sorted(set(r['label'] for r in recordings)))
assert len(recordings)>=9,'Need >=9 recordings for train/val/test split'

In [ ]:
# Deterministic recording-level split and training-only normalization.
def bench_split(seed):
    rng=np.random.default_rng(seed)
    # split each label to avoid losing instrument categories when feasible
    groups={}
    for r in recordings:groups.setdefault(r['label'],[]).append(r)
    train,val,test=[],[],[]
    for label,items in sorted(groups.items()):
        items=list(items);rng.shuffle(items)
        if len(items)>=3:
            nv=max(1,round(.2*len(items)));nt=max(1,round(.2*len(items)))
            val.extend(items[:nv]);test.extend(items[nv:nv+nt]);train.extend(items[nv+nt:])
        else:train.extend(items)
    if not val or not test:raise ValueError('Need >=3 recordings per label for stratified split')
    assert not ({r['id'] for r in train}&{r['id'] for r in test})
    return train,val,test

def bench_prepare(seed):
    train,val,test=bench_split(seed)
    tr=np.concatenate([r['features'] for r in train])
    mean=tr.mean(axis=0);std=tr.std(axis=0)+.01
    def make(items):
        xx,yy,ids=[],[],[]
        for r in items:
            a=np.clip((r['features']-mean)/std,-6,6)
            for start in range(0,len(a)-BENCH_CONTEXT-BENCH_H+1,24):
                xx.append(a[start:start+BENCH_CONTEXT])
                yy.append(a[start+BENCH_CONTEXT:start+BENCH_CONTEXT+BENCH_H])
                ids.append(r['id'])
        return torch.tensor(np.stack(xx),dtype=torch.float32),torch.tensor(np.stack(yy),dtype=torch.float32),ids
    return make(train),make(val),make(test),{'train':[r['id'] for r in train],
      'validation':[r['id'] for r in val],'test':[r['id'] for r in test]}

## Step 3 — Instantiate the persistent proposed dynamics and neural comparators

A history GRU initializes the global latent state once. During the next eight times, `torchdiffeq` integrates the spectral state and latent state jointly. We compare with a no-diffusion ablation, conventional Neural ODE, temporal CNN, GRU, and Transformer.

In [ ]:
import torch.nn.functional as F
class PersistentCoupled(nn.Module):
    def __init__(self, mode='coupled', bands=BENCH_BANDS, latent=12):
        super().__init__()
        self.mode=mode
        self.bands=bands;self.latent=latent
        self.history=nn.GRU(bands,latent,batch_first=True)
        self.local=nn.Sequential(nn.Conv1d(1,12,5,padding=2),nn.Tanh(),
                                 nn.Conv1d(12,1,3,padding=1))
        self.z_dynamics=nn.Sequential(nn.Linear(latent+1,24),nn.Tanh(),
                                      nn.Linear(24,latent))
        self.coupling=nn.Linear(latent,bands,bias=False)
        self.diffusion_raw=nn.Parameter(torch.tensor(-2.))
    def forward(self,t,state):
        u,z=state
        padded=F.pad(u[:,None,:],(1,1),mode='replicate')[:,0]
        lap=padded[:,:-2]-2*u+padded[:,2:]
        D=F.softplus(self.diffusion_raw)
        if self.mode=='no_diffusion':D=D*0
        du=self.local(u[:,None,:])[:,0]+self.coupling(z)+D*lap
        dz=self.z_dynamics(torch.cat([z,u.mean(-1,keepdim=True)],dim=-1))
        return du,dz
    def predict(self,x,dt,method='rk4',rtol=1e-4,atol=1e-5):
        _,z=self.history(x);z=z[-1]
        u=x[:,-1]
        t=torch.arange(BENCH_H+1,device=x.device,dtype=x.dtype)*dt
        opts={'step_size':dt/2} if method=='rk4' else None
        states=odeint(self,(u,z),t,method=method,options=opts,rtol=rtol,atol=atol)
        return states[0][1:].transpose(0,1)

class ConventionalNeuralODE(nn.Module):
    def __init__(self,bands=BENCH_BANDS,hidden=48):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(bands,hidden),nn.Tanh(),
                               nn.Linear(hidden,bands))
    def forward(self,t,u):return self.net(u)
    def predict(self,x,dt,method='rk4',rtol=1e-4,atol=1e-5):
        t=torch.arange(BENCH_H+1,device=x.device,dtype=x.dtype)*dt
        opts={'step_size':dt/2} if method=='rk4' else None
        return odeint(self,x[:,-1],t,method=method,options=opts,rtol=rtol,atol=atol)[1:].transpose(0,1)

class TemporalCNN(nn.Module):
    def __init__(self,bands=BENCH_BANDS,hidden=64):
        super().__init__()
        self.net=nn.Sequential(nn.Conv1d(bands,hidden,3,padding=1),nn.ReLU(),
                               nn.Conv1d(hidden,hidden,3,padding=1),nn.ReLU(),
                               nn.AdaptiveAvgPool1d(1))
        self.head=nn.Linear(hidden,BENCH_H*bands)
    def predict(self,x,dt,**kw):
        return self.head(self.net(x.transpose(1,2)).squeeze(-1)).reshape(-1,BENCH_H,x.shape[-1])

class GRUForecast(nn.Module):
    def __init__(self,bands=BENCH_BANDS,hidden=64):
        super().__init__()
        self.gru=nn.GRU(bands,hidden,batch_first=True)
        self.head=nn.Linear(hidden,BENCH_H*bands)
    def predict(self,x,dt,**kw):
        _,h=self.gru(x)
        return self.head(h[-1]).reshape(-1,BENCH_H,x.shape[-1])

class TinyTransformer(nn.Module):
    def __init__(self,bands=BENCH_BANDS,d_model=48):
        super().__init__()
        self.proj=nn.Linear(bands,d_model)
        self.pos=nn.Parameter(torch.zeros(1,BENCH_CONTEXT,d_model))
        layer=nn.TransformerEncoderLayer(d_model,4,dim_feedforward=96,
                 dropout=0,batch_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=1)
        self.head=nn.Linear(d_model,BENCH_H*bands)
    def predict(self,x,dt,**kw):
        h=self.encoder(self.proj(x)+self.pos)
        return self.head(h[:,-1]).reshape(-1,BENCH_H,x.shape[-1])

BENCH_MODELS={
 'coupled':lambda:PersistentCoupled('coupled'),
 'no_diffusion':lambda:PersistentCoupled('no_diffusion'),
 'neural_ode':ConventionalNeuralODE,
 'tcn':TemporalCNN,'gru':GRUForecast,'transformer':TinyTransformer,
}
BENCH_DT=BENCH_HOP/BENCH_SR
print('Parameters by architecture:')
for name,ctor in BENCH_MODELS.items():
    model=ctor()
    print(name,sum(p.numel() for p in model.parameters()))


## Step 4 — Define music-specific measurements

The evaluation reports all-horizon spectral MSE, high-frequency-band MSE, spectral-flux event F1 and transient localization. Only validation recordings calibrate the spectral-flux threshold; these proxy events are not professionally annotated note onsets.

In [ ]:
def bench_flux(x,context_last):
    previous=torch.cat([context_last[:,None,:],x[:,:-1]],dim=1)
    return F.relu(x-previous).sum(-1)
def bench_onset_metrics(pred,true,context_last,threshold):
    pf=bench_flux(pred,context_last).cpu().numpy()
    tf=bench_flux(true,context_last).cpu().numpy()
    tp=fp=fn=0
    offsets=[]
    for p,t in zip(pf,tf):
        pidx=np.where(p>threshold)[0].tolist()
        tidx=np.where(t>threshold)[0].tolist()
        used=set()
        for pi in pidx:
            eligible=[ti for ti in tidx if ti not in used and abs(ti-pi)<=1]
            if eligible:
                ti=min(eligible,key=lambda k:abs(k-pi))
                used.add(ti);tp+=1;offsets.append(abs(ti-pi))
            else:fp+=1
        fn+=len(tidx)-len(used)
    precision=tp/max(tp+fp,1);recall=tp/max(tp+fn,1)
    f1=2*precision*recall/max(precision+recall,1e-12)
    return {'onset_f1':f1,
            'transient_localization_mae_frames':float(np.mean(offsets)) if offsets else None,
            'reference_onset_events':tp+fn,'predicted_onset_events':tp+fp}
def bench_evaluate(pred,y,x,threshold):
    result={'mse':float((pred-y).square().mean()),
            'hf_mse':float((pred[:,:,-BENCH_BANDS//4:]-y[:,:,-BENCH_BANDS//4:]).square().mean()),
            'per_horizon_mse':(pred-y).square().mean((0,2)).cpu().tolist()}
    result.update(bench_onset_metrics(pred,y,x[:,-1],threshold))
    return result


## Step 5 — Train the proposed model and non-pretrained comparators

We match the recording splits, context length and training epoch budget across learned architectures. Parameter count, training time and memory are retained. We report solver tolerance sensitivity separately because integration accuracy can change results.

In [ ]:
def bench_fit_one(name,ctor,train,val,seed):
    torch.manual_seed(seed)
    model=ctor().to(device)
    optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
    train_x,train_y,_=train
    val_x,val_y,_=val
    loader=DataLoader(TensorDataset(train_x,train_y),batch_size=BENCH_BATCH,shuffle=True)
    best=float('inf');checkpoint=None
    if device.type=='cuda':torch.cuda.reset_peak_memory_stats()
    start=time.perf_counter()
    for epoch in range(BENCH_EPOCHS):
        model.train()
        for bx,by in loader:
            bx=bx.to(device);by=by.to(device)
            optimizer.zero_grad()
            pred=model.predict(bx,BENCH_DT)
            loss=(pred-by).square().mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            vloss=[]
            for bx,by in DataLoader(TensorDataset(val_x,val_y),batch_size=BENCH_BATCH*2):
                vloss.append(float((model.predict(bx.to(device),BENCH_DT)-by.to(device)).square().mean()))
            v=float(np.mean(vloss))
        if v<best:
            best=v;checkpoint={k:val.detach().cpu().clone() for k,val in model.state_dict().items()}
    train_s=time.perf_counter()-start
    model.load_state_dict(checkpoint)
    model.eval()
    peak=(torch.cuda.max_memory_allocated()/1024**2) if device.type=='cuda' else None
    return model,{'training_s':train_s,'validation_mse':best,
                  'parameters':sum(p.numel() for p in model.parameters()),
                  'peak_gpu_memory_mib':peak}

def bench_predict_batches(model,x):
    predictions=[]
    with torch.no_grad():
        for batch in x.split(BENCH_BATCH*2):
            predictions.append(model.predict(batch.to(device),BENCH_DT).cpu())
    return torch.cat(predictions)

def bench_timed_predictions(model,x):
    if device.type=='cuda':torch.cuda.synchronize()
    start=time.perf_counter()
    pred=bench_predict_batches(model,x)
    if device.type=='cuda':torch.cuda.synchronize()
    return pred,1000*(time.perf_counter()-start)/len(x)

In [ ]:
BENCH_ROWS=[]
BENCH_HORIZONS=[]
BENCH_SOLVER=[]
for seed in BENCH_SEEDS:
    train,val,test,split_ids=bench_prepare(seed)
    train_x,train_y,_=train;val_x,val_y,_=val;test_x,test_y,_=test
    # Threshold from validation ground truth only; no test calibration.
    threshold=float(torch.quantile(bench_flux(val_y,val_x[:,-1]).flatten(),.85))
    for name,ctor in BENCH_MODELS.items():
        model,stats=bench_fit_one(name,ctor,train,val,seed)
        pred,infer_ms=bench_timed_predictions(model,test_x)
        metric=bench_evaluate(pred,test_y,test_x,threshold)
        BENCH_ROWS.append({'seed':seed,'model':name,'inference_ms_per_window':infer_ms,
                           **stats,**{k:v for k,v in metric.items() if k!='per_horizon_mse'}})
        for h,value in enumerate(metric['per_horizon_mse'],1):
            BENCH_HORIZONS.append({'seed':seed,'model':name,'horizon':h,'mse':value})
        # Solver sensitivity on a fixed test minibatch (adaptive Dopri5 vs RK4)
        if name in ('coupled','no_diffusion','neural_ode'):
            bx=test_x[:min(8,len(test_x))].to(device)
            with torch.no_grad():
                ref=model.predict(bx,BENCH_DT,method='rk4')
                for tol in (1e-3,1e-5):
                    try:
                        alt=model.predict(bx,BENCH_DT,method='dopri5',rtol=tol,atol=tol/10)
                        delta=float((alt-ref).square().mean())
                    except Exception as exc:
                        delta=float('nan')
                        print('Solver warning:',name,seed,exc)
                    BENCH_SOLVER.append({'seed':seed,'model':name,'rtol':tol,
                                         'mse_vs_rk4':delta})
        print('Seed',seed,'model',name,'test MSE',round(metric['mse'],5))
        del model
        if device.type=='cuda':torch.cuda.empty_cache()
    # Non-neural baselines get identical test windows and onset threshold.
    for name,pred in (
        ('persistence',test_x[:,-1:].expand(-1,BENCH_H,-1)),
        ('linear_extrapolation',torch.stack(
          [test_x[:,-1]+(h+1)*(test_x[:,-1]-test_x[:,-2]) for h in range(BENCH_H)],dim=1))):
        metric=bench_evaluate(pred,test_y,test_x,threshold)
        BENCH_ROWS.append({'seed':seed,'model':name,'inference_ms_per_window':0.,
                           'training_s':0.,'validation_mse':None,
                           'parameters':0,'peak_gpu_memory_mib':0.,
                           **{k:v for k,v in metric.items() if k!='per_horizon_mse'}})
        for h,value in enumerate(metric['per_horizon_mse'],1):
            BENCH_HORIZONS.append({'seed':seed,'model':name,'horizon':h,'mse':value})
    print('Split:',{k:len(v) for k,v in split_ids.items()})

## Step 6 — Add three pretrained Hugging Face forecasting baselines

This is the primary new V2 benchmark extension. We evaluate **three distinct Hugging Face checkpoints**, with no training and exactly the same **512-step input** and **eight-step target** as the proposed continuous-time model:

1. `ibm-research/ttm-research-r2`, revision `main`: TinyTimeMixer (512 historical points, 96 generated points; we use the first eight). Its **research license is CC BY-NC-SA 4.0**.
2. `amazon/chronos-bolt-tiny`: pretrained direct quantile time-series forecaster, using the median quantile.
3. `amazon/chronos-t5-tiny`: pretrained probabilistic time-series forecaster, using the median of 20 generated samples.

**Fairness and resource policy:** We evaluate the **same small, fixed subset of 12 held-out windows and eight frequency bands** for all three pretrained models and compare the proposed model on that *exact* subset. This keeps Colab costs practical; it is a pilot, not a complete full-spectrum benchmark. For model cards and primary sources, see References. We do not insert zeros or interpolate short sequences to trick the 512-step TTM model.

We calculate mean square error, high-frequency-band MSE (in the chosen subset), and per-window inference time. Any model that fails to load or produce correctly shaped output is reported as `unavailable` with a reason, never replaced by a fabricated forecast.

In [ ]:
%pip -q install 'granite-tsfm' 'chronos-forecasting' 'transformers>=4.45,<5'

In [ ]:
# Zero-shot HF forecasting models: TTM-R2, Chronos-Bolt-Tiny, Chronos-T5-Tiny.
# This code is deliberately separated from training so that model download and compute costs are visible.
RUN_HF_FORECAST=True
HF_MAX_WINDOWS=12
HF_CHANNEL_INDEX=torch.linspace(0,BENCH_BANDS-1,8).round().long()
HF_RESULT_ROWS=[]
HF_PREDICTIONS={}

def hf_test_slice():
    # This data is strictly from unseen recording-level test records.
    seed=BENCH_SEEDS[0]
    train,val,test,split=bench_prepare(seed)
    xx,yy,_=test
    xx=xx[:HF_MAX_WINDOWS,:,HF_CHANNEL_INDEX].contiguous()
    yy=yy[:HF_MAX_WINDOWS,:,HF_CHANNEL_INDEX].contiguous()
    assert xx.shape[1]==512, ('TTM needs 512 genuine context frames',xx.shape)
    assert xx.shape[-1]==8 and yy.shape[1]==BENCH_H
    return xx,yy,seed


def run_hf_forecasters():
    import traceback, gc
    x,truth,seed=hf_test_slice()
    # Evaluate the proposed model on the EXACT SAME x/y sample, but preserve its full spectrum.
    train,val,test,split=bench_prepare(seed)
    test_full,test_full_y,_=test
    torch.manual_seed(seed)
    base,stats=bench_fit_one('coupled',BENCH_MODELS['coupled'],train,val,seed)
    infer_start=time.perf_counter()
    full=bench_predict_batches(base,test_full[:len(x)])
    proposed_inference_s=time.perf_counter()-infer_start
    proposed=full[:,:,HF_CHANNEL_INDEX]
    def record(name,pred,seconds,notes='',params=None):
        assert tuple(pred.shape)==tuple(truth.shape),(name,pred.shape,truth.shape)
        pred=pred.cpu().float()
        row={'model':name,'status':'evaluated',
             'test_mse':float((pred-truth).square().mean()),
             'upper_band_mse':float((pred[:,:,-2:]-truth[:,:,-2:]).square().mean()),
             'inference_s':seconds,'windows':len(truth),'bands':len(HF_CHANNEL_INDEX),
             'params':params,'notes':notes}
        HF_RESULT_ROWS.append(row);HF_PREDICTIONS[name]=pred
    record('PROPOSED persistent PDE-ODE',proposed,proposed_inference_s,'trained from scratch; full-spectrum model')
    del base;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

    if not RUN_HF_FORECAST:
        for model_id in ('ibm-research/ttm-research-r2','amazon/chronos-bolt-tiny','amazon/chronos-t5-tiny'):
            HF_RESULT_ROWS.append({'model':model_id,'status':'skipped','error':'RUN_HF_FORECAST=False'})
        return
    # Standalone dependencies are deliberately installed in the preceding cell.
    for model_id in ('ibm-research/ttm-research-r2','amazon/chronos-bolt-tiny','amazon/chronos-t5-tiny'):
        model=None
        started=time.perf_counter()
        try:
            if 'ttm-research-r2' in model_id:
                from tsfm_public.models.tinytimemixer import TinyTimeMixerForPrediction
                model=TinyTimeMixerForPrediction.from_pretrained(model_id,revision='main').to(device).eval()
                if getattr(model.config,'context_length',512)!=512:
                    raise ValueError('Checkpoint context length is not 512')
                # Frequency-token-enabled checkpoints may require a token. Reject if this
                # model cannot infer without a token; do not invent an audio frequency label.
                with torch.inference_mode():
                    pred=model(past_values=x.to(device)).prediction_outputs[:,:BENCH_H,:].cpu()
            else:
                from chronos import BaseChronosPipeline, ChronosPipeline
                # Univariate per frequency band. Same test records as multivariate coupled model.
                context=x.transpose(1,2).reshape(-1,512)
                if 'bolt' in model_id:
                    model=BaseChronosPipeline.from_pretrained(model_id,device_map=str(device))
                    with torch.inference_mode():
                        out=model.predict(context,prediction_length=BENCH_H)
                    # Chronos-Bolt predict -> (num_series, num_quantiles, horizon)
                    pred=out[:,out.shape[1]//2,:].reshape(len(x),8,BENCH_H).permute(0,2,1)
                else:
                    model=ChronosPipeline.from_pretrained(model_id,device_map=str(device),torch_dtype=torch.float32)
                    with torch.inference_mode():
                        out=model.predict(context,prediction_length=BENCH_H,num_samples=20)
                    # Chronos-T5 -> (num_series, num_samples, horizon)
                    pred=out.median(dim=1).values.reshape(len(x),8,BENCH_H).permute(0,2,1)
            record(model_id,pred,time.perf_counter()-started,
                   notes='zero-shot; TTM and Chronos pretrained on non-audio time series')
        except Exception as exc:
            HF_RESULT_ROWS.append({'model':model_id,'status':'unavailable',
                                   'error':type(exc).__name__+': '+str(exc)[:300]})
            print('Model unavailable:',model_id,type(exc).__name__,str(exc)[:300])
        finally:
            del model
            gc.collect()
            if torch.cuda.is_available():torch.cuda.empty_cache()

run_hf_forecasters()
print(pd.DataFrame(HF_RESULT_ROWS).to_string(index=False))

## Evaluation — Does the proposed method actually win?

The first evaluation is full-spectrum across recording-disjoint test splits. The second, focused on IBM TTM and the two Chronos models, uses the identical 12 held-out windows and eight frequency channels across all four methods. The proposed PDE–ODE is **highlighted by a thick hatched outline**. We will not infer superiority from any missing pretrained-model result.

In [ ]:
benchmark_df=pd.DataFrame(BENCH_ROWS)
horizon_df=pd.DataFrame(BENCH_HORIZONS)
solver_df=pd.DataFrame(BENCH_SOLVER)
numeric=['mse','hf_mse','onset_f1','training_s','inference_ms_per_window','parameters']
summary=benchmark_df.groupby('model')[numeric].agg(['mean','std']).round(6)
print('Cross-recording multi-seed benchmark:')
print(summary.to_string())
print('Note: onset metrics use spectral-flux proxy labels, not annotated music onsets.')
benchmark_df.to_csv('music_recording_level_raw_metrics.csv',index=False)
horizon_df.to_csv('music_recording_level_horizon_metrics.csv',index=False)
solver_df.to_csv('music_solver_sensitivity.csv',index=False)
summary.to_csv('music_recording_level_summary.csv')

fig,axes=plt.subplots(1,3,figsize=(17,4.5))
g=benchmark_df.groupby('model',as_index=False).agg(mse=('mse','mean'),hf=('hf_mse','mean'),onset=('onset_f1','mean'))
axes[0].bar(g.model,g.mse);axes[0].set_title('Unseen-recording spectral MSE ↓')
axes[1].bar(g.model,g.hf);axes[1].set_title('High-frequency detail MSE ↓')
axes[2].bar(g.model,g.onset);axes[2].set_title('Onset proxy F1 ↑')
for a in axes:a.tick_params(axis='x',rotation=55)
fig.tight_layout()
fig.savefig('music_recording_level_comparison.png',dpi=160)
plt.show()

fig,ax=plt.subplots(figsize=(10,4))
for name,frame in horizon_df.groupby('model'):
    means=frame.groupby('horizon').mse.mean()
    ax.plot(means.index,means.values,marker='o',label=name)
ax.set(xlabel='Future frame',ylabel='Held-out MSE ↓',title='Forecast error by horizon')
ax.legend(ncol=2,fontsize=8)
fig.tight_layout()
fig.savefig('music_recording_level_horizons.png',dpi=160)
plt.show()
Path('music_benchmark_protocol.json').write_text(json.dumps({
    'seeds':BENCH_SEEDS,'horizon':BENCH_H,'context':BENCH_CONTEXT,
    'epochs':BENCH_EPOCHS,'recordings':[r['id'] for r in recordings],
    'synthetic_fallback':not bool(paths),'split':'recording-level stratified',
    'warning':'Synthetic tracks do not establish real-audio genre generalization'
},indent=2))
print('Saved 4 CSVs, 2 PNGs, and protocol JSON')


In [ ]:
HF_DF=pd.DataFrame(HF_RESULT_ROWS)
HF_DF.to_csv('music_hf_three_foundation_models.csv',index=False)
valid_hf=HF_DF[HF_DF.status=='evaluated'].copy()
if not valid_hf.empty:
    fig,ax=plt.subplots(figsize=(12,4.5))
    bars=ax.bar(valid_hf.model.astype(str),valid_hf.test_mse,alpha=.65)
    for label,bar in zip(valid_hf.model,bars):
        if label=='PROPOSED persistent PDE-ODE':
            bar.set_edgecolor('black');bar.set_linewidth(3.5);bar.set_hatch('///')
            ax.annotate('PROPOSED', (bar.get_x()+bar.get_width()/2,bar.get_height()),
                        xytext=(0,9),textcoords='offset points',ha='center',weight='bold')
    ax.set(ylabel='Test MSE on identical 8 spectral bands (lower is better)',
           title='Proposed continuous-time method vs 3 HF pretrained forecasters')
    ax.tick_params(axis='x',rotation=25)
    fig.tight_layout();fig.savefig('music_hf_three_model_proposed_comparison.png',dpi=160);plt.show()
print('HF model status:',HF_DF[['model','status']].to_string(index=False))
if HF_DF.status.eq('evaluated').sum()>=4:
    print('Proposed method beats all three HF forecasters:',float(valid_hf.loc[valid_hf.model=='PROPOSED persistent PDE-ODE','test_mse'].iloc[0]) < float(valid_hf.loc[valid_hf.model!='PROPOSED persistent PDE-ODE','test_mse'].min()))
else:
    print('Incomplete HF benchmark: no three-model superiority claim can be made.')

In [ ]:
# Evidence-based conclusion for the three Hugging Face pretrained forecasting baselines.
print('EVIDENCE-BASED CONCLUSION')
evaluated=HF_DF[HF_DF.status=='evaluated']
proposed=evaluated[evaluated.model=='PROPOSED persistent PDE-ODE']
others=evaluated[evaluated.model!='PROPOSED persistent PDE-ODE']
if len(others)<3 or len(proposed)!=1:
    print('Incomplete pretrained-model comparison; cannot claim three-baseline superiority.')
else:
    score=float(proposed.test_mse.iloc[0])
    print('Proposed test MSE:',score,'; best pretrained HF MSE:',float(others.test_mse.min()))
    print('Proposed beats all three:',bool(score<float(others.test_mse.min())))
print('Future research: licensed real music, recording/artist splits, equal time budgets, calibration.')


## Discussion and limitations

The results generated above must be interpreted relative to the proposed method, not simply by selecting the lowest bar. We report rejected proposals and unavailable pretrained checkpoints as distinct statuses.

## Conclusion and future research

We interpret a smaller test error as an empirical advantage **only for the measured task and data split**, not as proof that a PDE describes music physically. The agent should distinguish (i) a proposed model that improves prediction, (ii) a change that merely increases capacity, and (iii) effects caused by information leakage or unequal context length. We retain rejected proposals, out-of-domain failures and compute overhead in the report.

### Conclusion

This executable study tests a falsifiable continuous-time modelling or weight-space-repair hypothesis. **The scientific conclusion must be based on the generated tables and plots rather than on the expected-outcomes paragraph.** In particular, no improvement is asserted before executing the relevant models and checking held-out data.

### Future research

1. Repeat over at least 10 independent seeds with confidence intervals and artist-/recording-disjoint test data.
2. Compare with compute-matched classical, deep and pretrained time-series forecasting models.
3. Examine spectral diffusion under noisy, irregularly sampled real music and identify when it destroys transients.
4. Investigate JAX/Equinox/Diffrax implementations for compiled coupled simulation and high-order sensitivity analysis.
5. For WSP, expand from eight hand-selected kernel entries to learned localization and editing of multiple network layers; test proposal validity and reproducibility.
6. Register hypotheses and statistical decision rules before tuning, publish complete per-seed metrics and model provenance.

## References

1. M. Habiba and B. Pearlmutter, *Continuous Convolutional Neural Networks: Coupled Neural PDE and ODE*, 2021. https://arxiv.org/abs/2111.00343
2. R. T. Q. Chen et al., *Neural Ordinary Differential Equations*, NeurIPS 2018. https://arxiv.org/abs/1806.07366
3. `torchdiffeq`, differentiable ordinary differential equation solvers. https://github.com/rtqichen/torchdiffeq
4. R. Takehi et al., *Sparks of Weight-Space Programming*, 2026. https://rikiyat.github.io/blog/weight-space-programming/
5. V. Ekambaram et al., *Tiny Time Mixers*, NeurIPS 2024. https://arxiv.org/abs/2401.03955
6. A. Ansari et al., *Chronos: Learning the Language of Time Series*, 2024. https://arxiv.org/abs/2403.07815
7. IBM Research, TTM-R2 model card (research license). https://huggingface.co/ibm-research/ttm-research-r2
8. Amazon, Chronos-Bolt-Tiny model card. https://huggingface.co/amazon/chronos-bolt-tiny
9. Amazon, Chronos-T5-Tiny model card. https://huggingface.co/amazon/chronos-t5-tiny
10. Hugging Face, Qwen2.5 models. https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct and https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
11. Hugging Face, SmolLM2 Instruct. https://huggingface.co/HuggingFaceTB/SmolLM2-1.7B-Instruct

### Reproducibility and licensing statement

We use synthetic signals only unless the operator uploads recordings that they are permitted to analyse. The IBM `ibm-research/ttm-research-r2` model has a **CC BY-NC-SA 4.0 research-use** license; do not assume it is suitable for commercial deployment. TTM-R2 was pretrained on time-series frequencies unlike 16-ms audio features; the comparison measures **distribution-shift generalisation**, not in-domain foundation-model quality. A commercial-use alternative is IBM's separately licensed Granite TTM release.